In [8]:
#seq2seq英译法案例
#案例步骤：
#1.数据预处理：读取数据，构建词汇表，转换为索引
#2.模型定义：定义编码器和解码器
#3.训练模型：定义损失函数和优化器，训练模型
#4.评估模型：测试模型性能，进行翻译

import re #正则表达式模块
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import time

import random #随机数生成器
import matplotlib.pyplot as plt

device=torch.device("cuda" if torch.cuda.is_available() else "cpu") #设备适配
SOS_token=0 #设置开始标记和结束标记
EOS_token=1

max_length=10 #设置最大句子长度
path='./data/eng-fra-v2.txt'




In [9]:
#数据预处理
#读取文档数据，构建词汇表和索引映射
eng_sentences=[] #英语句子列表
fra_sentences=[]
with open(path,'r',encoding='utf-8') as f:
    sentences=f.read().strip().split('\n')
    print(sentences[:5]) #打印前5行数据
    for sentence in sentences:
        eng,fra=sentence.split('\t')
        eng_sentences.append(eng)
        fra_sentences.append(fra)
    print(eng_sentences[:5])
    print(len(fra_sentences))

#构建两个句子列表的字典对应
eng_fra_dict={}
for i in range(len(eng_sentences)):
    eng_fra_dict.update({eng_sentences[i]:fra_sentences[i]})
# print(eng_fra_dict)

#构建词汇表和索引映射
eng_words=[]
fra_words=[]
for sentence in eng_sentences:
    eng_words.extend(sentence.split(' '))
print(eng_words[:5])

for sentence in fra_sentences:
    fra_words.extend(sentence.split(' '))
print(fra_words[:5])

eng_vocab=list(set(eng_words)) #英语词汇表
fra_vocab=list(set(fra_words))
print(len(eng_vocab))
print(len(fra_vocab))

eng_word_idx={'SOS':SOS_token,'EOS':EOS_token} #英语词汇索引映射
fra_word_idx={'SOS':SOS_token,'EOS':EOS_token} #法语词汇
i=2
j=2
for word in eng_vocab:
    eng_word_idx[word]=i
    i+=1
for word in fra_vocab:
    fra_word_idx[word]=j
    j+=1
# print(fra_word_idx)
# print(fra_word_idx.items())

eng_idx_word={i:w for w,i in eng_word_idx.items()}
fra_idx_word={i:w for w,i in fra_word_idx.items()}
print(eng_idx_word)

eng_seq_idx=[]
fra_seq_idx=[]
#将句子转换为索引序列
for sentence in eng_sentences:
    tem=[eng_word_idx[w] for w in sentence.split(' ')]
    eng_seq_idx.append(tem)
print(eng_seq_idx[:5])

for sentence in fra_sentences:
    tem=[fra_word_idx[w] for w in sentence.split(' ')]
    fra_seq_idx.append(tem)
print(fra_seq_idx[:5])

['i m .\tj ai ans .', 'i m ok .\tje vais bien .', 'i m ok .\tca va .', 'i m fat .\tje suis gras .', 'i m fat .\tje suis gros .']
['i m .', 'i m ok .', 'i m ok .', 'i m fat .', 'i m fat .']
63594
['i', 'm', '.', 'i', 'm']
['j', 'ai', 'ans', '.', 'je']
2801
4343
{0: 'SOS', 1: 'EOS', 2: 'unforgettable', 3: 'sea', 4: 'traffic', 5: 'czech', 6: 'chasing', 7: 'breathing', 8: 'leader', 9: 'recognize', 10: 'overworked', 11: 'custom', 12: 'of', 13: 'pressing', 14: 'germans', 15: 'renown', 16: 'weird', 17: 'insane', 18: 'thanksgiving', 19: 'misconduct', 20: 'read', 21: 'ticklish', 22: 'manipulative', 23: 'early', 24: 'husband', 25: 'getting', 26: 'way', 27: 'socks', 28: 'rusty', 29: 'beggar', 30: 'deluding', 31: 'morons', 32: 'restaurant', 33: 'jam', 34: 'toed', 35: 'putting', 36: 'distantly', 37: 'came', 38: 'sunbathing', 39: 'appalling', 40: 'devastated', 41: 'outlaw', 42: 'belly', 43: 'warming', 44: 'the', 45: 'unsung', 46: 'single', 47: 'gathering', 48: 'creative', 49: 'clever', 50: 'fifty', 

In [10]:
#构建数据集
eng_fra_list=list(eng_fra_dict.items())

#定义数据集类
class SeqDataset(Dataset):
    def __init__(self,eng_fra_list):
        self.eng_fra_pairs=eng_fra_list
        self.length=len(eng_fra_list)
    def __len__(self):
        return self.length
    def __getitem__(self,idx):
        eng=self.eng_fra_pairs[idx][0]
        fra=self.eng_fra_pairs[idx][1]
        eng_idx=[eng_word_idx[w] for w in eng.split(' ')]
        eng_idx.append(eng_word_idx['EOS']) #添加结束标记
        fra_idx=[fra_word_idx[w] for w in fra.split(' ')]
        fra_idx.append(fra_word_idx['EOS'])
        x=torch.tensor(eng_idx)
        y=torch.tensor(fra_idx)
        return x,y

#实例化数据集和数据加载器
dataset=SeqDataset(eng_fra_list)
dataloader=DataLoader(dataset,batch_size=1,shuffle=True)
for x,y in dataloader:
    print(x.shape)
    print(y.shape)


torch.Size([1, 7])
torch.Size([1, 7])
torch.Size([1, 9])
torch.Size([1, 7])
torch.Size([1, 7])
torch.Size([1, 8])
torch.Size([1, 7])
torch.Size([1, 9])
torch.Size([1, 7])
torch.Size([1, 9])
torch.Size([1, 7])
torch.Size([1, 6])
torch.Size([1, 9])
torch.Size([1, 10])
torch.Size([1, 9])
torch.Size([1, 9])
torch.Size([1, 9])
torch.Size([1, 9])
torch.Size([1, 9])
torch.Size([1, 10])
torch.Size([1, 8])
torch.Size([1, 9])
torch.Size([1, 8])
torch.Size([1, 9])
torch.Size([1, 8])
torch.Size([1, 8])
torch.Size([1, 7])
torch.Size([1, 8])
torch.Size([1, 7])
torch.Size([1, 6])
torch.Size([1, 5])
torch.Size([1, 5])
torch.Size([1, 6])
torch.Size([1, 6])
torch.Size([1, 8])
torch.Size([1, 8])
torch.Size([1, 5])
torch.Size([1, 4])
torch.Size([1, 7])
torch.Size([1, 7])
torch.Size([1, 8])
torch.Size([1, 7])
torch.Size([1, 8])
torch.Size([1, 10])
torch.Size([1, 10])
torch.Size([1, 9])
torch.Size([1, 8])
torch.Size([1, 8])
torch.Size([1, 8])
torch.Size([1, 7])
torch.Size([1, 8])
torch.Size([1, 7])
torch.Si

In [32]:
#定义编码器和解码器
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed=nn.Embedding(len(eng_word_idx),128) #词嵌入层
        self.gru=nn.GRU(128,256,1,batch_first=True)
    def forward(self,x,h0):
        x=self.embed(x)
        output,hn=self.gru(x,h0)
        return output,hn
    def init_hidden(self,batch_size):
        return torch.zeros(1,batch_size,256,device=device)

#检验编码器
encoder=Encoder().to(device)
for x,y in dataloader:
    x=x.to(device)
    y=y.to(device)
    h0=encoder.init_hidden(x.size(0))
    output,hn=encoder(x,h0)
    print(output.shape)
    print(hn.shape)





torch.Size([1, 7, 256])
torch.Size([1, 1, 256])
torch.Size([1, 8, 256])
torch.Size([1, 1, 256])
torch.Size([1, 7, 256])
torch.Size([1, 1, 256])
torch.Size([1, 6, 256])
torch.Size([1, 1, 256])
torch.Size([1, 8, 256])
torch.Size([1, 1, 256])
torch.Size([1, 10, 256])
torch.Size([1, 1, 256])
torch.Size([1, 7, 256])
torch.Size([1, 1, 256])
torch.Size([1, 8, 256])
torch.Size([1, 1, 256])
torch.Size([1, 5, 256])
torch.Size([1, 1, 256])
torch.Size([1, 6, 256])
torch.Size([1, 1, 256])
torch.Size([1, 8, 256])
torch.Size([1, 1, 256])
torch.Size([1, 7, 256])
torch.Size([1, 1, 256])
torch.Size([1, 8, 256])
torch.Size([1, 1, 256])
torch.Size([1, 5, 256])
torch.Size([1, 1, 256])
torch.Size([1, 8, 256])
torch.Size([1, 1, 256])
torch.Size([1, 5, 256])
torch.Size([1, 1, 256])
torch.Size([1, 5, 256])
torch.Size([1, 1, 256])
torch.Size([1, 7, 256])
torch.Size([1, 1, 256])
torch.Size([1, 8, 256])
torch.Size([1, 1, 256])
torch.Size([1, 5, 256])
torch.Size([1, 1, 256])
torch.Size([1, 6, 256])
torch.Size([1, 